In [0]:
%sql
-- Vilka distanser finns i time_based (distanslopp)?
SELECT event_distance_length, COUNT(*) as antal
FROM marathos.silver.obt
WHERE performance_seconds IS NOT NULL
GROUP BY event_distance_length
ORDER BY antal DESC
LIMIT 20

In [0]:
print(df.count())

In [0]:
# Vilka event_type finns? (distance vs time)
from pyspark.sql.functions import col, when

df_types = df.withColumn(
    "marathon_type",
    when(col("performance_seconds").isNotNull(), "time_based")
    .otherwise("distance_based")
)
display(df_types.groupBy("marathon_type").count())

In [0]:
%sql
-- Vilka tider finns i distance_based (tidslopp)?
SELECT event_distance_length, COUNT(*) as antal
FROM marathos.silver.obt
WHERE performance_km IS NOT NULL
GROUP BY event_distance_length
ORDER BY antal DESC
LIMIT 20

In [0]:
%sql

-- Kolla vad distance_based faktiskt har
SELECT event_distance_length, performance_km, performance_seconds, COUNT(*) as antal
FROM marathos.silver.obt
WHERE event_distance_length LIKE '%h'
GROUP BY event_distance_length, performance_km, performance_seconds
LIMIT 20



In [0]:
%sql
-- Kolla athlete_performance direkt för tidslopp
SELECT event_distance_length, athlete_performance, athlete_average_speed, COUNT(*) as antal
FROM marathos.silver.obt
WHERE event_distance_length LIKE '%h'
GROUP BY event_distance_length, athlete_performance, athlete_average_speed
LIMIT 20

In [0]:
%sql
-- Visa alla unika event_distance_length värden
SELECT DISTINCT event_distance_length
FROM marathos.silver.obt
ORDER BY event_distance_length
LIMIT 50

In [0]:
%sql
-- Hur många km vs mi lopp?
SELECT 
  CASE 
    WHEN event_distance_length LIKE '%mi' THEN 'miles'
    WHEN event_distance_length LIKE '%km' THEN 'kilometers'
    ELSE 'other'
  END as race_type,
  COUNT(*) as antal
FROM marathos.silver.obt
GROUP BY race_type

In [0]:
%sql

SELECT 
  event_distance_length,
  MIN(performance_seconds) as snabbaste_sekunder,
  ROUND(MIN(performance_seconds) / 3600.0, 2) as snabbaste_timmar,
  ROUND(AVG(athlete_average_speed), 2) as snitt_hastighet_kmh,
  COUNT(*) as antal
FROM marathos.silver.obt
WHERE event_distance_length IN ('50km', '100km', '50mi', '100mi')
GROUP BY event_distance_length
ORDER BY event_distance_length

In [0]:
%sql
SELECT 
  event_distance_length,
  COUNT(*) as total,
  COUNT(CASE WHEN performance_seconds = 0 THEN 1 END) as nollor,
  MIN(CASE WHEN performance_seconds > 0 THEN performance_seconds END) as snabbaste_riktig,
  ROUND(MIN(CASE WHEN performance_seconds > 0 THEN performance_seconds END) / 3600.0, 2) as snabbaste_timmar
FROM marathos.silver.obt
WHERE event_distance_length IN ('50km', '100km', '50mi', '100mi')
GROUP BY event_distance_length
ORDER BY event_distance_length

In [0]:
%sql
-- Bekräfta att nollorna är borta
SELECT 
  event_distance_length,
  COUNT(*) as total,
  MIN(performance_seconds) as snabbaste_sekunder,
  ROUND(MIN(performance_seconds) / 3600.0, 2) as snabbaste_timmar,
  ROUND(AVG(athlete_average_speed), 2) as snitt_hastighet_kmh
FROM marathos.silver.obt
WHERE event_distance_length IN ('50km', '100km', '50mi', '100mi')
GROUP BY event_distance_length
ORDER BY event_distance_length

In [0]:
#show all tables incloded in gold schema
display(spark.sql("SHOW TABLES IN marathos.gold"))

In [0]:
for t in ["dim_athlete","dim_event","dim_time","fct_results","mart_fastest_athletes_km","mart_fastest_athletes_mi","mart_top_countries_km","mart_top_countries_mi"]:
    print(f"\n=== marathos.gold.{t} ===")
    spark.sql(f"DESCRIBE marathos.gold.{t}").show(truncate=False)